# SoMark OXR-1.0 (A100 default · D-FINE Layout + vLLM OXR + SoMark-compatible Parse API)

Serves SoMark OXR-1.0 (1.3B BF16 document parser) as three loopback services on one GPU: D-FINE layout analysis (`127.0.0.1:8091`), the OXR VLM via `vllm==0.25.1` + the out-of-tree plugin shipped with the weights (`127.0.0.1:8090`), and the SoMark-compatible parse API (`/v1/parse/sync`, `/v1/parse/async`, `/demo`, `/docs` on `127.0.0.1:8000`). Caddy enforces Basic Auth / Bearer token, and SoMark SDK/CLI `api_key` form fields are verified fail-closed. Weights and three uv virtualenvs live on local SSD; compiler caches persist to Google Drive.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "oxr")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = os.environ.get("CT_WORK_DIR", "/content/ct")             # logs + runtime configs (0700)
CACHE_DIR = os.environ.get("CT_CACHE_DIR", "")
BIN_DIR = os.environ.get("CT_BIN_DIR", "/usr/local/bin")
os.makedirs(WORK, mode=0o700, exist_ok=True)
os.chmod(WORK, 0o700)
os.makedirs(BIN_DIR, exist_ok=True)
if BIN_DIR not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = BIN_DIR + os.pathsep + os.environ.get("PATH", "")
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to <WORK>/<name>.log (0600)."""
    lp = f"{WORK}/{name}.log"
    fd = os.open(lp, os.O_WRONLY | os.O_CREAT | os.O_APPEND, 0o600)
    os.chmod(lp, 0o600)
    with os.fdopen(fd, "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

CADDY_VERSION = "2.10.2"

def ct_write_private_text(path, content):
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, "w", encoding="utf-8") as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())

def ct_proc_start_time(pid):
    try:
        fields = open(f"/proc/{int(pid)}/stat", "r", encoding="utf-8", errors="replace").read().rsplit(")", 1)[1].split()
        if fields[0] == "Z":
            return None
        return fields[19]
    except (OSError, IndexError, ValueError):
        return None

def ct_stop_marked_proc(name, timeout=10):
    import signal
    marker = f"{WORK}/{name}.proc.json"
    if not os.path.exists(marker):
        return
    try:
        info = json.loads(open(marker, "r", encoding="utf-8").read())
        pid = int(info.get("pid") or 0)
        start_time = info.get("start_time")
    except (OSError, ValueError, TypeError):
        pid, start_time = 0, None
    if pid > 0 and start_time and ct_proc_start_time(pid) == start_time:
        for sig in (signal.SIGTERM, signal.SIGKILL):
            if ct_proc_start_time(pid) != start_time:
                break
            try:
                os.killpg(os.getpgid(pid), sig)
            except (ProcessLookupError, PermissionError, OSError):
                try:
                    os.kill(pid, sig)
                except (ProcessLookupError, OSError):
                    break
            deadline = time.monotonic() + timeout
            while time.monotonic() < deadline:
                if ct_proc_start_time(pid) != start_time:
                    break
                time.sleep(0.1)
    try:
        os.unlink(marker)
    except OSError:
        pass

def ct_spawn_marked(name, args, env=None, cwd=None):
    ct_stop_marked_proc(name)
    proc = ct_spawn(name, args, env=env, cwd=cwd)
    marker = f"{WORK}/{name}.proc.json"
    ct_write_private_text(marker, json.dumps({"pid": proc.pid, "start_time": ct_proc_start_time(proc.pid)}))
    return proc

def _ct_file_sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def ct_ensure_caddy():
    import hashlib, platform, tarfile, tempfile
    arch = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")
    dest = f"{BIN_DIR}/caddy"
    status_label = "miss"
    manifest = f"{CACHE_DIR}/manifests/caddy.json" if CACHE_DIR else ""
    if os.path.exists(dest):
        if not manifest:
            return dest
        try:
            meta = json.loads(open(manifest, "r", encoding="utf-8").read())
            if meta.get("artifact") == "caddy" and meta.get("version") == CADDY_VERSION and meta.get("sha256") == _ct_file_sha256(dest):
                print("CT_CACHE_EVENT artifact=caddy status=hit", flush=True)
                return dest
        except (OSError, ValueError):
            pass
        try:
            os.unlink(dest)
        except OSError:
            pass
        status_label = "corrupt_rebuilt"
    asset = f"caddy_{CADDY_VERSION}_linux_{arch}.tar.gz"
    checks_asset = f"caddy_{CADDY_VERSION}_checksums.txt"
    base = f"https://github.com/caddyserver/caddy/releases/download/v{CADDY_VERSION}/"
    os.makedirs(BIN_DIR, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        tgz = os.path.join(tmp, asset)
        with urllib.request.urlopen(base + asset, timeout=60) as src, open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        checks = urllib.request.urlopen(base + checks_asset, timeout=60).read().decode("utf-8")
        expected = next((line.split()[0] for line in checks.splitlines() if line.split() and line.split()[-1].lstrip("*") == asset), None)
        algo = {64: hashlib.sha256, 128: hashlib.sha512}.get(len(expected or ""))
        if algo is None or algo(open(tgz, "rb").read()).hexdigest() != expected:
            raise RuntimeError(f"Caddy checksum verification failed for {asset}")
        partial = f"{dest}.{os.getpid()}.partial"
        try:
            with tarfile.open(tgz) as tar:
                entry = tar.getmember("caddy")
                with tar.extractfile(entry) as src, open(partial, "wb") as out:
                    shutil.copyfileobj(src, out)
            os.chmod(partial, 0o755)
            os.replace(partial, dest)
        finally:
            if os.path.exists(partial):
                os.unlink(partial)
    if manifest:
        os.makedirs(os.path.dirname(manifest), exist_ok=True)
        tmp_m = f"{manifest}.{os.getpid()}.tmp"
        with open(tmp_m, "w", encoding="utf-8") as mf:
            mf.write(json.dumps({
                "artifact": "caddy",
                "version": CADDY_VERSION,
                "sha256": _ct_file_sha256(dest),
                "bin_path": dest,
                "verified_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }, indent=2) + "\n")
        os.replace(tmp_m, manifest)
        print(f"CT_CACHE_EVENT artifact=caddy status={status_label}", flush=True)
    return dest

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None, heartbeat_interval=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    last_heartbeat = started
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        now_m = time.monotonic()
        if heartbeat is not None and (heartbeat_interval is None or (now_m - last_heartbeat) >= heartbeat_interval):
            last_heartbeat = now_m
            try:
                heartbeat(int(now_m - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- OXR repo + three uv venvs (vLLM 0.25.1 / D-FINE / parse API) + OXR-1.0 weights on local SSD
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import hashlib

OXR_REPO_URL = "https://github.com/SoMarkAI/OXR.git"
OXR_COMMIT = os.environ.get("CT_OXR_COMMIT", "b2ee4ac12ed220bf7fefa74951babf0d3b05e766").strip()
OXR_MODEL_REPO = "SoMarkAI/OXR-1.0"
OXR_MODEL_REVISION = os.environ.get("CT_OXR_MODEL_REVISION", "c708407f9d2b4df658f2543d9132b62bb7221df4").strip()
VLLM_PORT = int(os.environ.get("CT_OXR_VLLM_PORT", "8090"))     # Colab itself owns :8080
DFINE_PORT = int(os.environ.get("CT_OXR_DFINE_PORT", "8091"))
VLLM_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"
DFINE_URL = f"http://127.0.0.1:{DFINE_PORT}"

OXR_WORK = Path("/content/oxr")
OXR_REPO = OXR_WORK / "repo"
OXR_MODEL = OXR_WORK / "model" / "OXR-1.0"
OXR_RUN = OXR_WORK / "run"
VLLM_VENV, DFINE_VENV, API_VENV = OXR_WORK / "vllm-venv", OXR_WORK / "dfine-venv", OXR_WORK / "api-venv"
VLLM_PY, DFINE_PY, API_PY = (str(v / "bin" / "python") for v in (VLLM_VENV, DFINE_VENV, API_VENV))
for _d in (OXR_WORK, OXR_MODEL, OXR_RUN, OXR_RUN / "outputs"):
    _d.mkdir(parents=True, exist_ok=True)

def olog(msg):
    print(time.strftime("[%F %T] [oxr] ") + msg, flush=True)

# BF16 GPU sizing: 1.3B BF16 weights are ~3.1 GB; vLLM and D-FINE share one GPU.
_gpu_line = ct_gpu_stat("name,memory.total,compute_cap").splitlines()[0]
if _gpu_line == "cpu/no-gpu":
    raise RuntimeError("OXR needs an NVIDIA BF16 GPU (A100 / L4 / H100 / G4); got a CPU runtime")
_gpu_name, _gpu_mem, _gpu_cc = (x.strip() for x in _gpu_line.split(","))
GPU_TOTAL_GIB = float(_gpu_mem.split()[0]) / 1024
if float(_gpu_cc) < 8.0:
    raise RuntimeError(f"OXR needs BF16 (compute capability >= 8.0); {_gpu_name} is sm{_gpu_cc}")
GPU_MEM_UTIL = float(os.environ.get("CT_OXR_GPU_MEM_UTIL") or "0.70")
MAX_NUM_SEQS = int(os.environ.get("CT_OXR_MAX_NUM_SEQS") or (16 if GPU_TOTAL_GIB < 32 else 32 if GPU_TOTAL_GIB < 64 else 64))
MAX_MODEL_LEN = int(os.environ.get("CT_OXR_MAX_MODEL_LEN") or "16384")
OXR_CONCURRENCY = int(os.environ.get("CT_OXR_CONCURRENCY") or MAX_NUM_SEQS)
if not 0.2 <= GPU_MEM_UTIL <= 0.9:
    raise RuntimeError(f"CT_OXR_GPU_MEM_UTIL must be within 0.2-0.9, got {GPU_MEM_UTIL}")
olog(f"GPU {_gpu_name} {GPU_TOTAL_GIB:.1f} GiB sm{_gpu_cc} | vLLM util={GPU_MEM_UTIL} "
     f"max_num_seqs={MAX_NUM_SEQS} max_model_len={MAX_MODEL_LEN} api_concurrency={OXR_CONCURRENCY}")

# Compiler caches on local SSD, archived to Drive per environment + GPU architecture.
(OXR_WORK / "colab_cache.py").write_text('"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}，已用时 {time.monotonic()-started:.0f} 秒\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'缓存归档包含不安全的路径或文件类型\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'缓存目标包含符号链接\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: 已在本地恢复，跳过重复传输\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: 从网盘读取归档（{stat.st_size/1e6:.1f} MB）\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: 在本地 SSD 解包\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: 归档恢复完成\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: 归档不可用，回退旧缓存（{error}）\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: 首次迁移旧版网盘缓存（小文件较多时耗时较长）\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: 旧缓存迁移失败，回退按需生成（{error}）\')\n    log(f\'{local.name}: 无持久缓存，将由服务按需生成\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: 无缓存文件，跳过归档\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: 内容未变化，跳过上传\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: 本地打包 {len(files)} 个文件\', pack)\n            _progress(log, f\'{local.name}: 保存归档到网盘（{staged.stat().st_size/1e6:.1f} MB）\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: 网盘归档已更新\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n', encoding="utf-8")
sys.path.insert(0, str(OXR_WORK))
from colab_cache import restore_cache, save_cache
CACHE_KEY = "oxr-vllm0.25.1-torch2.11-py312-sm" + _gpu_cc.replace(".", "")
CACHE_ARCHIVES = Path(DATA_DIR) / "cache-archives"
CACHE_DIRS = {name: OXR_WORK / name for name in ("vllm-cache", "triton-cache", "flashinfer-cache", "inductor-cache")}
CACHE_ENV = {
    "VLLM_CACHE_ROOT": str(CACHE_DIRS["vllm-cache"]),
    "TRITON_CACHE_DIR": str(CACHE_DIRS["triton-cache"]),
    "FLASHINFER_WORKSPACE_BASE": str(CACHE_DIRS["flashinfer-cache"]),
    "TORCHINDUCTOR_CACHE_DIR": str(CACHE_DIRS["inductor-cache"]),
}
for _local in CACHE_DIRS.values():
    try:
        restore_cache(_local, Path(DATA_DIR) / _local.name, CACHE_ARCHIVES, CACHE_KEY, olog)
    except (OSError, ValueError) as err:
        olog(f"Cache restore warning ({_local.name}): {err}; continuing")

if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)
UV = shutil.which("uv") or "uv"

def run_logged(cmd, log_path, env=None, cwd=None):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"\n$ {' '.join(str(x) for x in cmd)}\n")
        f.flush()
        res = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=cwd, text=True)
    if res.returncode != 0:
        tail = "".join(open(log_path, errors="replace").readlines()[-45:])
        raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(str(x) for x in cmd[:5])}\n{tail}")

def _venv_ready(venv, signature, probe):
    marker = venv / ".ct_oxr_ready"
    py = venv / "bin" / "python"
    if marker.is_file() and marker.read_text() == signature and py.is_file():
        return subprocess.run([str(py), "-c", probe], capture_output=True, timeout=300).returncode == 0
    return False

def prepare_repo():
    log_path = OXR_WORK / "repo-setup.log"
    head = subprocess.run(["git", "-C", str(OXR_REPO), "rev-parse", "HEAD"], capture_output=True, text=True)
    if head.returncode == 0 and head.stdout.strip() == OXR_COMMIT:
        olog(f"Reusing OXR source {OXR_COMMIT[:12]}")
        return
    if not (OXR_REPO / ".git").is_dir():
        shutil.rmtree(OXR_REPO, ignore_errors=True)
        run_logged(["git", "init", "-q", str(OXR_REPO)], log_path)
        run_logged(["git", "-C", str(OXR_REPO), "remote", "add", "origin", OXR_REPO_URL], log_path)
    run_logged(["git", "-C", str(OXR_REPO), "fetch", "-q", "--depth", "1", "origin", OXR_COMMIT], log_path)
    run_logged(["git", "-C", str(OXR_REPO), "checkout", "-q", "--force", "--detach", "FETCH_HEAD"], log_path)
    olog(f"OXR source ready at {OXR_COMMIT[:12]}")

def prepare_vllm_env():
    signature = "vllm==0.25.1 transformers==5.13.1 py3.12\n"
    if _venv_ready(VLLM_VENV, signature, "import vllm, transformers"):
        olog("Reusing vLLM 0.25.1 virtualenv")
        return
    log_path = OXR_WORK / "vllm-env.log"
    olog(f"Installing vLLM 0.25.1 + Transformers 5.13.1 (torch 2.11 cu130; log: {log_path})")
    run_logged([UV, "venv", "--python", "3.12", "--seed", "--clear", str(VLLM_VENV)], log_path)
    run_logged([UV, "pip", "install", "--python", VLLM_PY, "vllm==0.25.1", "transformers==5.13.1"], log_path)
    (VLLM_VENV / ".ct_oxr_ready").write_text(signature)
    olog("vLLM environment ready")

def prepare_dfine_env():
    reqs = OXR_REPO / "services" / "layout_analyze" / "requirements.txt"
    signature = "torch==2.11.0 torchvision==0.26.0\n" + reqs.read_text()
    if _venv_ready(DFINE_VENV, signature, "import torch, cv2, fastapi, uvicorn"):
        olog("Reusing D-FINE virtualenv")
        return
    log_path = OXR_WORK / "dfine-env.log"
    olog(f"Installing D-FINE layout environment (log: {log_path})")
    run_logged([UV, "venv", "--python", "3.12", "--clear", str(DFINE_VENV)], log_path)
    run_logged([UV, "pip", "install", "--python", DFINE_PY, "torch==2.11.0", "torchvision==0.26.0",
                "-r", str(reqs)], log_path)
    (DFINE_VENV / ".ct_oxr_ready").write_text(signature)
    olog("D-FINE environment ready")

def prepare_api_env():
    constraints = OXR_REPO / "docker" / "requirements.allinone.txt"
    signature = OXR_COMMIT + "\n" + constraints.read_text()
    if _venv_ready(API_VENV, signature, "import oxr.server.app, uvicorn"):
        olog("Reusing OXR parse API virtualenv")
        return
    log_path = OXR_WORK / "api-env.log"
    olog(f"Installing OXR parse API (log: {log_path})")
    run_logged([UV, "venv", "--python", "3.12", "--clear", str(API_VENV)], log_path)
    run_logged([UV, "pip", "install", "--python", API_PY, "-c", str(constraints), str(OXR_REPO)], log_path)
    (API_VENV / ".ct_oxr_ready").write_text(signature)
    olog("OXR parse API environment ready")

OXR_REQUIRED = ("config.json", "model.safetensors", "tokenizer.json", "chat_template.jinja",
                "plugins/vllm/pyproject.toml", "layout_analysis/last.pth", "layout_analysis/metadata.json")

def _model_complete():
    for rel in OXR_REQUIRED:
        path = OXR_MODEL / rel
        if not path.is_file() or path.stat().st_size == 0:
            return False
        with path.open("rb") as handle:
            if handle.read(64).startswith(b"version https://git-lfs"):
                return False
    return (OXR_MODEL / "model.safetensors").stat().st_size > 1_000_000_000

def prepare_model():
    marker = OXR_MODEL / ".ct_model_ready"
    if marker.is_file() and marker.read_text().strip() == OXR_MODEL_REVISION and _model_complete():
        olog(f"Reusing local SSD weights {OXR_MODEL}")
        return
    log_path = OXR_WORK / "model-download.log"
    olog(f"Downloading {OXR_MODEL_REPO}@{OXR_MODEL_REVISION[:12]} -> {OXR_MODEL} (log: {log_path})")
    env = {**os.environ, "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0", "HF_HUB_ENABLE_HF_TRANSFER": "0"}
    script = ("import sys; from huggingface_hub import snapshot_download; "
              "snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], local_dir=sys.argv[3], max_workers=8, "
              "ignore_patterns=['.eval_results/*'])")
    try:
        run_logged([UV, "run", "--no-project", "--python", "3.12", "--with", "huggingface_hub[hf_xet]>=0.34",
                    "python", "-c", script, OXR_MODEL_REPO, OXR_MODEL_REVISION, str(OXR_MODEL)], log_path, env=env)
    finally:
        for p in ("/root/.cache/huggingface/xet", str(OXR_MODEL / ".cache")):
            shutil.rmtree(p, ignore_errors=True)
    if not _model_complete():
        raise RuntimeError(f"OXR-1.0 download incomplete in {OXR_MODEL}")
    marker.write_text(OXR_MODEL_REVISION + "\n")
    olog("OXR-1.0 weights ready on local SSD")

def install_plugin():
    plugin_src = OXR_MODEL / "plugins" / "vllm"
    digest = hashlib.sha256(b"".join(p.read_bytes() for p in sorted(plugin_src.rglob("*.py")))
                            + (plugin_src / "pyproject.toml").read_bytes()).hexdigest()
    marker = VLLM_VENV / ".ct_oxr_plugin"
    if marker.is_file() and marker.read_text() == digest and subprocess.run(
            [VLLM_PY, "-c", "import oxr_vllm"], capture_output=True, timeout=300).returncode == 0:
        return
    staged = OXR_WORK / "plugin-src"
    shutil.rmtree(staged, ignore_errors=True)
    shutil.copytree(plugin_src, staged, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.egg-info", "build", "dist"))
    run_logged([UV, "pip", "install", "--python", VLLM_PY, "--no-deps", "--reinstall-package", "oxr-vllm-plugin",
                str(staged)], OXR_WORK / "vllm-env.log")
    marker.write_text(digest)
    olog("OXR vLLM plugin installed")

prepare_repo()
with ThreadPoolExecutor(max_workers=4) as pool:
    jobs = {name: pool.submit(fn) for name, fn in (("vllm", prepare_vllm_env), ("dfine", prepare_dfine_env),
                                                   ("api", prepare_api_env), ("model", prepare_model))}
    while not all(job.done() for job in jobs.values()):
        for job in jobs.values():
            if job.done() and job.exception():
                raise job.exception()
        olog("Setup heartbeat | " + " ".join(f"{n}={'done' if j.done() else 'running'}" for n, j in jobs.items())
             + f" | free_ssd={shutil.disk_usage('/content').free / 1e9:.1f} GB")
        time.sleep(30)
    for job in jobs.values():
        job.result()
install_plugin()
olog(f"Setup complete: repo={OXR_REPO} model={OXR_MODEL} data={DATA_DIR}")


In [ ]:
# @title 2. Service -- D-FINE :8091 -> vLLM OXR :8090 -> SoMark-compatible parse API :8000 (all 127.0.0.1)
import hashlib
(OXR_WORK / "oxr_auth_app.py").write_text('"""ASGI wrapper for the SoMark OXR parse API on Colab (embedded by templates/build_notebooks.py).\n\nCaddy authenticates every public request (Basic Auth / session cookie / ``Authorization: Bearer``)\nand adds ``X-CT-Proxy-Auth: <secret>`` on those authenticated branches. SoMark\'s official SDK/CLI\ncannot send headers; it only puts ``api_key`` into the form body of ``POST /v1/parse/*``. Caddy\nforwards exactly those three POST routes without the trusted header, and this wrapper then requires\nthe form field ``api_key`` to equal the runtime API token (constant time, fail-closed).\n\nThe wrapper also adds ``GET /health`` (aggregated D-FINE + vLLM + API readiness) and redirects\n``GET /`` to the bundled ``/demo`` UI. It imports only the standard library at module import so it\ncan be unit-tested locally; the upstream OXR app is imported lazily on the Colab VM.\n"""\nfrom __future__ import annotations\n\nimport asyncio\nimport json\nimport os\nimport secrets\nimport urllib.error\nimport urllib.parse\nimport urllib.request\n\nFORM_AUTH_PATHS = frozenset({"/v1/parse/sync", "/v1/parse/async", "/v1/parse/async_check"})\nPROXY_HEADER = b"x-ct-proxy-auth"\nMAX_FORM_AUTH_BODY = int(os.environ.get("CT_OXR_MAX_BODY_BYTES", str(512 * 1024 * 1024)))\n\n\ndef _header(scope, name: bytes) -> bytes:\n    for key, value in scope.get("headers") or ():\n        if key.lower() == name:\n            return value\n    return b""\n\n\ndef _equal(given, expected: str) -> bool:\n    if not expected or not isinstance(given, (str, bytes)):\n        return False\n    if isinstance(given, str):\n        given = given.encode("utf-8", errors="replace")\n    return secrets.compare_digest(given, expected.encode("utf-8"))\n\n\ndef _multipart_boundary(content_type: str) -> bytes:\n    for part in content_type.split(";")[1:]:\n        key, _, value = part.strip().partition("=")\n        if key.strip().lower() == "boundary":\n            value = value.strip()\n            if len(value) >= 2 and value[0] == value[-1] == \'"\':\n                value = value[1:-1]\n            return value.encode("latin-1", errors="replace")\n    return b""\n\n\ndef _part_field_name(headers: bytes):\n    """Return (name, has_filename) from one multipart part header block."""\n    for line in headers.split(b"\\r\\n"):\n        key, _, value = line.partition(b":")\n        if key.strip().lower() != b"content-disposition":\n            continue\n        name, has_filename = None, False\n        for item in value.split(b";")[1:]:\n            k, _, v = item.strip().partition(b"=")\n            k = k.strip().lower()\n            v = v.strip()\n            if len(v) >= 2 and v[:1] == v[-1:] == b\'"\':\n                v = v[1:-1]\n            if k == b"name":\n                name = v.decode("utf-8", errors="replace")\n            elif k in (b"filename", b"filename*"):\n                has_filename = True\n        return name, has_filename\n    return None, False\n\n\ndef extract_form_field(content_type: str, body: bytes, field: str = "api_key"):\n    """Return the first non-file form value named ``field`` or ``None``.\n\n    Supports ``multipart/form-data`` (SoMark parse uploads) and\n    ``application/x-www-form-urlencoded`` (SoMark ``async_check``). Large file parts are scanned with\n    ``bytes.find`` and never copied.\n    """\n    media_type = content_type.split(";", 1)[0].strip().lower()\n    if media_type == "application/x-www-form-urlencoded":\n        try:\n            values = urllib.parse.parse_qs(body.decode("utf-8"), keep_blank_values=True, max_num_fields=1000)\n        except (UnicodeDecodeError, ValueError):\n            return None\n        found = values.get(field)\n        return found[0] if found else None\n    if media_type != "multipart/form-data":\n        return None\n    boundary = _multipart_boundary(content_type)\n    if not boundary or len(boundary) > 200:\n        return None\n    delimiter = b"--" + boundary\n    position = body.find(delimiter)\n    while position != -1:\n        start = position + len(delimiter)\n        if body[start:start + 2] == b"--":\n            return None\n        if body[start:start + 2] == b"\\r\\n":\n            start += 2\n        header_end = body.find(b"\\r\\n\\r\\n", start)\n        if header_end == -1 or header_end - start > 16384:\n            return None\n        name, has_filename = _part_field_name(body[start:header_end])\n        content_start = header_end + 4\n        next_position = body.find(b"\\r\\n" + delimiter, content_start)\n        if next_position == -1:\n            return None\n        if name == field and not has_filename:\n            return body[content_start:next_position].decode("utf-8", errors="replace")\n        position = next_position + 2\n    return None\n\n\nasync def _send_json(send, status: int, payload: dict, extra_headers=()) -> None:\n    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")\n    headers = [(b"content-type", b"application/json; charset=utf-8"),\n               (b"content-length", str(len(body)).encode()),\n               (b"cache-control", b"no-store"), *extra_headers]\n    await send({"type": "http.response.start", "status": status, "headers": headers})\n    await send({"type": "http.response.body", "body": body})\n\n\ndef _probe(url: str, timeout: float = 5.0):\n    try:\n        with urllib.request.urlopen(url, timeout=timeout) as response:\n            raw = response.read(65536)\n            return response.status, raw\n    except urllib.error.HTTPError as error:\n        return error.code, b""\n    except (OSError, ValueError):\n        return 0, b""\n\n\ndef upstream_health(dfine_url: str, vllm_url: str, model_name: str = "oxr") -> dict:\n    """Return readiness of the D-FINE layout service and the vLLM OXR model service."""\n    status = {"dfine": False, "vllm": False}\n    code, raw = _probe(dfine_url.rstrip("/") + "/health")\n    if code == 200:\n        try:\n            status["dfine"] = json.loads(raw).get("status") == "healthy"\n        except (ValueError, AttributeError):\n            status["dfine"] = False\n    code, raw = _probe(vllm_url.rstrip("/") + "/models")\n    if code == 200:\n        try:\n            status["vllm"] = any(item.get("id") == model_name for item in json.loads(raw).get("data") or ())\n        except (ValueError, AttributeError, TypeError):\n            status["vllm"] = False\n    return status\n\n\nclass OxrAuthMiddleware:\n    """Fail-closed SoMark ``api_key`` form authentication + /health + / -> /demo redirect."""\n\n    def __init__(self, app, api_token: str, proxy_secret: str, *, dfine_url: str = "", vllm_url: str = "",\n                 model_name: str = "oxr", max_body: int = MAX_FORM_AUTH_BODY, health=None):\n        self.app = app\n        self.api_token = api_token or ""\n        self.proxy_secret = proxy_secret or ""\n        self.dfine_url = dfine_url\n        self.vllm_url = vllm_url\n        self.model_name = model_name\n        self.max_body = max_body\n        self.health = health or (lambda: upstream_health(self.dfine_url, self.vllm_url, self.model_name))\n\n    async def __call__(self, scope, receive, send):\n        if scope.get("type") != "http":\n            return await self.app(scope, receive, send)\n        path, method = scope.get("path", ""), scope.get("method", "GET").upper()\n        if path == "/health" and method in ("GET", "HEAD"):\n            status = await asyncio.to_thread(self.health)\n            ok = all(status.values())\n            return await _send_json(send, 200 if ok else 503, {"ok": ok, "api": True, **status})\n        if path == "/" and method in ("GET", "HEAD"):\n            await send({"type": "http.response.start", "status": 302,\n                        "headers": [(b"location", b"/demo"), (b"content-length", b"0")]})\n            return await send({"type": "http.response.body", "body": b""})\n        if path not in FORM_AUTH_PATHS or method != "POST":\n            return await self.app(scope, receive, send)\n        if self.proxy_secret and _equal(_header(scope, PROXY_HEADER), self.proxy_secret):\n            return await self.app(scope, receive, send)\n        if not self.api_token or not self.proxy_secret:\n            return await _send_json(send, 401, {"code": 1107, "message": "API key authentication is not configured"})\n        chunks, total = [], 0\n        declared = _header(scope, b"content-length")\n        if declared.isdigit() and int(declared) > self.max_body:\n            return await _send_json(send, 413, {"code": 1133, "message": "Request body too large"})\n        while True:\n            message = await receive()\n            if message.get("type") == "http.disconnect":\n                return None\n            chunk = message.get("body", b"")\n            total += len(chunk)\n            if total > self.max_body:\n                return await _send_json(send, 413, {"code": 1133, "message": "Request body too large"})\n            chunks.append(chunk)\n            if not message.get("more_body"):\n                break\n        body = b"".join(chunks)\n        content_type = _header(scope, b"content-type").decode("latin-1", errors="replace")\n        if not _equal(extract_form_field(content_type, body), self.api_token):\n            return await _send_json(send, 401, {"code": 1107, "message": "Invalid api_key"},\n                                    extra_headers=[(b"www-authenticate", b\'Bearer realm="oxr"\')])\n        replayed = False\n\n        async def replay():\n            nonlocal replayed\n            if not replayed:\n                replayed = True\n                return {"type": "http.request", "body": body, "more_body": False}\n            return await receive()\n\n        return await self.app(scope, replay, send)\n\n\ndef create_app():\n    from oxr.server.app import app as oxr_app  # imported on the Colab VM only\n\n    return OxrAuthMiddleware(\n        oxr_app,\n        os.environ.get("CT_API_TOKEN", ""),\n        os.environ.get("CT_OXR_PROXY_SECRET", ""),\n        dfine_url=os.environ.get("OXR__LAYOUT_ANALYZE_MODEL__URL", "http://127.0.0.1:8091"),\n        vllm_url=os.environ.get("OXR__OXR_MODEL__URL", "http://127.0.0.1:8090/v1"),\n        model_name=os.environ.get("OXR__OXR_MODEL__MODEL_NAME", "oxr"),\n    )\n\n\ndef __getattr__(name):\n    if name == "app":\n        value = create_app()\n        globals()["app"] = value\n        return value\n    raise AttributeError(name)\n', encoding="utf-8")
# Caddy adds this header on its authenticated branches; SoMark form-key requests arrive without it.
OXR_PROXY_SECRET = hashlib.sha256(("ct-oxr-proxy-v1:" + API_TOKEN).encode()).hexdigest()

def start_dfine():
    return ct_spawn_marked("dfine", [DFINE_PY, "-m", "uvicorn", "dfine_la.api:app",
                              "--app-dir", str(OXR_REPO / "services" / "layout_analyze"),
                              "--host", "127.0.0.1", "--port", str(DFINE_PORT), "--workers", "1"],
                    cwd=str(OXR_RUN), env={**CACHE_ENV,
                        "MODEL_PATH": str(OXR_MODEL / "layout_analysis" / "last.pth"),
                        "MODEL_METADATA_PATH": str(OXR_MODEL / "layout_analysis" / "metadata.json"),
                        "DFINE_DEVICE": "cuda", "DFINE_CUDA_GRAPH": "1", "DFINE_PARALLEL_PREPROCESS": "1"})

def start_vllm():
    return ct_spawn_marked("vllm", [str(VLLM_VENV / "bin" / "vllm"), "serve", str(OXR_MODEL),
                             "--served-model-name", "oxr", "--host", "127.0.0.1", "--port", str(VLLM_PORT),
                             "--trust-remote-code", "--chat-template", str(OXR_MODEL / "chat_template.jinja"),
                             "--dtype", "bfloat16", "--max-model-len", str(MAX_MODEL_LEN),
                             "--gpu-memory-utilization", str(GPU_MEM_UTIL), "--max-num-seqs", str(MAX_NUM_SEQS),
                             "--attention-config", '{"backend":"TRITON_ATTN"}'],
                    cwd=str(OXR_RUN), env={**CACHE_ENV, "VLLM_PLUGINS": "oxr", "CUDA_HOME": "/usr/local/cuda",
                                           # FlashInfer JIT (sampling kernels) needs the venv's ninja + nvcc.
                                           "PATH": f"{VLLM_VENV / 'bin'}:/usr/local/cuda/bin:{os.environ.get('PATH', '')}"})

def start_service():
    return ct_spawn_marked("service", [API_PY, "-m", "uvicorn", "oxr_auth_app:app", "--app-dir", str(OXR_WORK),
                                "--host", "127.0.0.1", "--port", str(SERVICE_PORT), "--workers", "1"],
                    cwd=str(OXR_RUN), env={
                        "CT_API_TOKEN": API_TOKEN, "CT_OXR_PROXY_SECRET": OXR_PROXY_SECRET,
                        "OXR__LAYOUT_ANALYZE_MODEL__URL": DFINE_URL,
                        "OXR__OXR_MODEL__URL": VLLM_URL, "OXR__OXR_MODEL__MODEL_NAME": "oxr",
                        "OXR__OXR_MODEL__MAX_CONCURRENCY": str(OXR_CONCURRENCY),
                        "OXR__PIPELINE__IMAGE_OUTPUT_DIR": str(OXR_RUN / "outputs")})

def _dfine_healthy():
    try:
        with urllib.request.urlopen(DFINE_URL + "/health", timeout=10) as resp:
            return json.loads(resp.read()).get("status") == "healthy"
    except Exception:
        return False

# Reused VM/kernel: PROCS is reset by cell 0, so stop stale processes via their pid markers.
for _name in ("service", "vllm", "dfine"):
    ct_stop_marked_proc(_name, timeout=30)
# D-FINE first so vLLM's memory profiling sees its CUDA graphs already resident.
PROCS["dfine"] = (start_dfine(), start_dfine)
_deadline = time.monotonic() + 900
while not _dfine_healthy():
    if PROCS["dfine"][0].poll() is not None:
        raise RuntimeError("D-FINE exited during startup:\n" + ct_tail("dfine", 60))
    if time.monotonic() > _deadline:
        raise RuntimeError("D-FINE did not become healthy within 900s:\n" + ct_tail("dfine", 60))
    time.sleep(3)
olog(f"D-FINE healthy on {DFINE_URL} | GPU: {ct_gpu_stat()}")
PROCS["vllm"] = (start_vllm(), start_vllm)
PROCS["service"] = (start_service(), start_service)
olog(f"vLLM (:{VLLM_PORT}) and parse API (:{SERVICE_PORT}) starting")


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)

def start_caddy():
    caddy_bin = ct_ensure_caddy()
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output([caddy_bin, "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            header_up X-CT-Proxy-Auth {OXR_PROXY_SECRET}
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @ct_form_auth {{
        method POST
        path /v1/parse/sync /v1/parse/async /v1/parse/async_check
    }}
    handle @ct_form_auth {{
        reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            header_up Host {{upstream_hostport}}
            header_up -Authorization
            header_up -X-CT-Proxy-Auth
            flush_interval -1
        }}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    ct_write_private_text(f"{WORK}/Caddyfile", caddyfile)
    return ct_spawn_marked("caddy", [caddy_bin, "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CLOUDFLARED_VERSION = '2026.11.0'
CLOUDFLARED_SHA256 = {
    'amd64': '51f7c68b57ed7269164075d7e789efce5b59355982f06392deeb037d982a51fd',
    'arm64': 'abe54a006acf9faf2d19ee9d953cb0d0804e92f59c5382a162e57b84d086f56f',
}


def _write_private_text(path, content):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, 'w', encoding='utf-8') as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())


def _touch_private_log(path):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    os.close(fd)


def _file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _check_cached_binary(destination, artifact, version, cache_dir):
    if not destination.exists():
        return None
    if not cache_dir:
        return 'hit'
    manifest = Path(cache_dir) / 'manifests' / f'{artifact}.json'
    if not manifest.is_file():
        return 'miss'
    try:
        meta = json.loads(manifest.read_text(encoding='utf-8'))
        if (
            meta.get('artifact') == artifact
            and meta.get('version') == version
            and meta.get('sha256') == _file_sha256(destination)
        ):
            return 'hit'
    except (OSError, ValueError):
        pass
    destination.unlink(missing_ok=True)
    return 'corrupt_rebuilt'


def _write_manifest(cache_dir, artifact, version, destination):
    if not cache_dir:
        return
    manifest_dir = Path(cache_dir) / 'manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest = manifest_dir / f'{artifact}.json'
    tmp = manifest.with_name(f'.{artifact}.{os.getpid()}.tmp')
    payload = {
        'artifact': artifact,
        'version': version,
        'sha256': _file_sha256(destination),
        'bin_path': str(destination),
        'verified_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    tmp.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
    os.replace(tmp, manifest)


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version, *, cache_dir=None):
    destination = Path(destination)
    cache_state = _check_cached_binary(destination, 'frpc', version, cache_dir)
    if cache_state == 'hit':
        if cache_dir:
            print(f'CT_CACHE_EVENT artifact=frpc status=hit', flush=True)
        return str(destination)
    status_label = cache_state or 'miss'
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            partial=destination.with_name(f'{destination.name}.{os.getpid()}.partial')
            try:
                with tar.extractfile(entry) as source, open(partial, 'wb') as out_bin:
                    shutil.copyfileobj(source, out_bin)
                partial.chmod(0o755)
                os.replace(partial, destination)
            finally:
                partial.unlink(missing_ok=True)
    _write_manifest(cache_dir, 'frpc', version, destination)
    if cache_dir:
        print(f'CT_CACHE_EVENT artifact=frpc status={status_label}', flush=True)
    return str(destination)


def install_cloudflared(destination, *, cache_dir=None, fetch_fn=None):
    destination = Path(destination)
    cf_arch = {'x86_64': 'amd64', 'aarch64': 'arm64'}.get(platform.machine(), 'amd64')
    url = f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-{cf_arch}'
    expected_sha = CLOUDFLARED_SHA256.get(cf_arch)
    if not expected_sha:
        raise ValueError(f'Unsupported cloudflared architecture: {platform.machine()}')
    if cache_dir:
        cache_state = _check_cached_binary(destination, 'cloudflared', CLOUDFLARED_VERSION, cache_dir)
        if cache_state == 'hit':
            print('CT_CACHE_EVENT artifact=cloudflared status=hit', flush=True)
            return str(destination)
        status_label = cache_state or 'miss'
        if destination.exists() and _file_sha256(destination) != expected_sha:
            destination.unlink(missing_ok=True)
            status_label = 'corrupt_rebuilt'
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
        _write_manifest(cache_dir, 'cloudflared', CLOUDFLARED_VERSION, destination)
        print(f'CT_CACHE_EVENT artifact=cloudflared status={status_label}', flush=True)
        return str(destination)
    if fetch_fn is not None:
        return fetch_fn(str(destination), url)
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx.get('WORK') or os.environ.get('CT_WORK_DIR') or '/content/ct')
    work.mkdir(parents=True, exist_ok=True)
    bin_dir_explicit = ctx.get('BIN_DIR') or os.environ.get('CT_BIN_DIR') or ''
    bin_dir = Path(bin_dir_explicit or '/usr/local/bin')
    cache_dir = ctx.get('CACHE_DIR') or os.environ.get('CT_CACHE_DIR') or ''
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if ctx.get('ct_stop_marked_proc') is not None:
        stop_fn = ctx['ct_stop_marked_proc']
        stop_fn('tunnel-process')
    elif marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        _touch_private_log(work/(name+'.log'))
        proc=ctx['ct_spawn'](name,args)
        _write_private_text(marker, json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        _write_private_text(ca_path, os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        _write_private_text(config_path, render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        frpc_name = f"frpc-{os.environ['CT_FRP_VERSION']}"
        frpc_dest = (bin_dir / frpc_name) if bin_dir_explicit else ((work / frpc_name) if (work / frpc_name).exists() else (bin_dir / frpc_name))
        binary=install_frpc(frpc_dest, os.environ['CT_FRP_VERSION'], cache_dir=cache_dir or None)
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        binary=install_cloudflared(bin_dir / 'cloudflared', cache_dir=cache_dir or None, fetch_fn=ctx.get('ct_fetch_binary'))
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see {work}/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- all three services healthy, real parse warmup, auth checks, then watchdog
import urllib.parse, uuid
LOCAL = f"http://127.0.0.1:{SERVICE_PORT}"
CADDY_LOCAL = f"http://127.0.0.1:{PUBLIC_PORT}"
UA = "Mozilla/5.0 (X11; Linux x86_64) colabllm-oxr"

def oxr_http(url, *, method="GET", headers=None, body=None, timeout=30):
    req = urllib.request.Request(url, data=body, method=method, headers={"User-Agent": UA, **(headers or {})})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status, resp.read()
    except urllib.error.HTTPError as err:
        return err.code, err.read()

def oxr_form(fields, files=()):
    boundary = "----ctoxr" + uuid.uuid4().hex
    parts = []
    for key, value in fields:
        parts.append(f'--{boundary}\r\nContent-Disposition: form-data; name="{key}"\r\n\r\n{value}\r\n'.encode())
    for key, name, ctype, data in files:
        parts.append(f'--{boundary}\r\nContent-Disposition: form-data; name="{key}"; filename="{name}"\r\n'
                     f'Content-Type: {ctype}\r\n\r\n'.encode() + data + b"\r\n")
    parts.append(f"--{boundary}--\r\n".encode())
    return b"".join(parts), {"Content-Type": f"multipart/form-data; boundary={boundary}"}

def oxr_vllm_ready():
    try:
        code, raw = oxr_http(VLLM_URL + "/models", timeout=10)
        return code == 200 and any(m.get("id") == "oxr" for m in json.loads(raw).get("data") or ())
    except Exception:
        return False

def oxr_stack_health(base=LOCAL, headers=None):
    code, raw = oxr_http(base + "/health", headers=headers, timeout=15)
    return code, (json.loads(raw) if raw[:1] == b"{" else {})

deadline = time.monotonic() + 2400
last_notice = 0
while True:
    for _name in ("dfine", "vllm", "service"):
        if PROCS[_name][0].poll() is not None:
            raise RuntimeError(f"{_name} exited during startup:\n" + ct_tail(_name, 80))
    try:
        code, info = oxr_stack_health()
        if code == 200 and info.get("ok"):
            break
    except Exception:
        info = {}
    if time.monotonic() > deadline:
        raise RuntimeError("OXR stack not ready within 2400s:\n" + ct_tail("vllm", 60) + ct_tail("service", 30))
    if time.monotonic() - last_notice >= 30:
        last_line = (ct_tail("vllm", 3).strip().splitlines() or ["starting..."])[-1]
        olog(f"Waiting for OXR stack {info or ''} | GPU: {ct_gpu_stat()} | vllm: {last_line[:110]}")
        last_notice = time.monotonic()
    time.sleep(5)
olog(f"All services healthy (D-FINE, vLLM, parse API) | GPU: {ct_gpu_stat()}")

# Real end-to-end warmup through the form-key path (layout -> recognition -> markdown).
_warm_png = OXR_RUN / "warmup.png"
subprocess.run([API_PY, "-c", (
    "import sys, cv2, numpy as np\n"
    "img = np.full((520, 1300, 3), 255, np.uint8)\n"
    "cv2.putText(img, 'OXR Warmup Report', (60, 110), cv2.FONT_HERSHEY_DUPLEX, 2.0, (0, 0, 0), 3)\n"
    "for i, line in enumerate(['Colab deployment readiness check.', 'Layout analysis, recognition and reading order.',"
    " 'Total: 42 documents parsed successfully.']):\n"
    "    cv2.putText(img, line, (60, 220 + 90 * i), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 0, 0), 2)\n"
    "cv2.imwrite(sys.argv[1], img)\n"), str(_warm_png)], check=True, timeout=120)
_body, _hdrs = oxr_form([("api_key", API_TOKEN), ("output_formats", "markdown"), ("output_formats", "json")],
                        [("file", "warmup.png", "image/png", _warm_png.read_bytes())])
_t0 = time.monotonic()
code, raw = oxr_http(LOCAL + "/v1/parse/sync", method="POST", headers=_hdrs, body=_body, timeout=600)
try:
    _warm = json.loads(raw)
except ValueError:
    _warm = {}
if code != 200 or _warm.get("code") != 0:
    raise RuntimeError(f"OXR warmup parse failed: HTTP {code} {raw[:400]!r}\n" + ct_tail("service", 40))
_md = ((_warm.get("data") or {}).get("result") or {}).get("outputs", {}).get("markdown", "")
olog(f"Warmup parse OK in {time.monotonic() - _t0:.1f}s ({len(_md)} markdown chars): {_md[:120]!r}")
if not _md.strip():
    olog("Warmup warning: markdown is empty; check vllm/dfine logs")

for _local in CACHE_DIRS.values():
    try:
        save_cache(_local, CACHE_ARCHIVES, CACHE_KEY, olog)
    except (OSError, ValueError) as err:
        olog(f"Cache archive save warning ({_local.name}): {err}")

def oxr_verify_auth(base):
    """No credentials / wrong Bearer / wrong form api_key -> 401; Bearer and form api_key -> accepted."""
    code, _ = oxr_http(base + "/health")
    if code != 401:
        raise RuntimeError(f"Unauthenticated GET {base}/health returned {code}, expected 401")
    code, _ = oxr_http(base + "/health", headers={"Authorization": "Bearer invalid-token-probe"})
    if code != 401:
        raise RuntimeError(f"Invalid Bearer on {base}/health returned {code}, expected 401")
    code, _ = oxr_stack_health(base, {"Authorization": f"Bearer {API_TOKEN}"})
    if code != 200:
        raise RuntimeError(f"Bearer GET {base}/health returned {code}, expected 200")
    probe = urllib.parse.urlencode({"task_id": "ct-auth-probe", "api_key": "invalid-key-probe"}).encode()
    form = {"Content-Type": "application/x-www-form-urlencoded"}
    code, _ = oxr_http(base + "/v1/parse/async_check", method="POST", headers=form, body=probe)
    if code != 401:
        raise RuntimeError(f"Invalid form api_key on {base}/v1/parse/async_check returned {code}, expected 401")
    probe = urllib.parse.urlencode({"task_id": "ct-auth-probe", "api_key": API_TOKEN}).encode()
    code, _ = oxr_http(base + "/v1/parse/async_check", method="POST", headers=form, body=probe)
    if code != 200:
        raise RuntimeError(f"Valid form api_key on {base}/v1/parse/async_check returned {code}, expected 200")

if AUTH == "caddy":
    oxr_verify_auth(CADDY_LOCAL)
    olog("Local Caddy auth verified (Basic/Bearer/SoMark form api_key)")
ct_public_gate(PUBLIC_URL, lambda: oxr_stack_health(PUBLIC_URL, {"Authorization": f"Bearer {API_TOKEN}"})[0],
               lambda: oxr_verify_auth(PUBLIC_URL), olog, timeout=180, interval=8)
olog(f"Public endpoint verified: {PUBLIC_URL}/demo")
if AUTH == "caddy" and not CREDS_FROM_ENV:   # started from the Colab UI: show generated credentials
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

def oxr_heartbeat(uptime):
    try:
        _, info = oxr_stack_health()
    except Exception as err:
        info = {"error": type(err).__name__}
    olog(f"heartbeat | up={uptime}s | {info} | GPU: {ct_gpu_stat()}")

ct_watchdog(oxr_vllm_ready, start_vllm, olog, service_key="vllm", log_name="vllm",
            dead_markers=("EngineDeadError", "EngineCore encountered a fatal error"),
            heartbeat=oxr_heartbeat, heartbeat_interval=240)
